# Question 1 — Conceptual: Best Subset vs. Forward and Backward Stepwise Selection
**ISLP, Chapter 6, Conceptual Exercise 1 (p. 283)**

**Author:** Brandon Perkins  
**Course:** DDS-8555 Predictive Analysis — Week 3  

---
We perform best subset, forward stepwise, and backward stepwise selection on a single data set. For each
approach we obtain $p+1$ models containing $0, 1, 2, \dots, p$ predictors.

This notebook states the answer to each part and then **verifies the claims empirically** on simulated data.

## (a) Which model with $k$ predictors has the smallest training RSS?

**Answer: best subset selection.**

Best subset selection fits **all** $\binom{p}{k}$ models of size $k$ and keeps the one with the lowest
training RSS. Forward stepwise only considers models that add one variable to its chosen $(k-1)$-variable
model, and backward stepwise only considers models that drop one variable from its chosen
$(k+1)$-variable model. Both therefore search a restricted path of $k$-variable models that is a
*subset* of what best subset examines. The best subset model can never be beaten on training RSS, so its
training RSS is **less than or equal to** that of the two stepwise models. Ties occur when the greedy path
happens to land on the global best subset.

## (b) Which model with $k$ predictors has the smallest test RSS?

**Answer: we cannot know in advance — any of the three may win.**

Training RSS is a deterministic, in-sample quantity, but test RSS depends on how well the selected model
generalizes to unseen data. Best subset has the lowest *training* RSS by construction, yet that same
exhaustive search over $\sum_k \binom{p}{k} = 2^p$ models gives it the greatest opportunity to overfit
noise, which inflates variance. A stepwise method that searches less aggressively may select a model that
generalizes better. The winner is data-dependent and must be determined by cross-validation or a
validation set rather than by theory (James et al., 2023).

## (c) True or False

| # | Statement | Answer |
|---|---|---|
| i | Predictors in the $k$-variable **forward** model are a subset of the $(k+1)$-variable **forward** model | **True** |
| ii | Predictors in the $k$-variable **backward** model are a subset of the $(k+1)$-variable **backward** model | **True** |
| iii | Predictors in the $k$-variable **backward** model are a subset of the $(k+1)$-variable **forward** model | **False** |
| iv | Predictors in the $k$-variable **forward** model are a subset of the $(k+1)$-variable **backward** model | **False** |
| v | Predictors in the $k$-variable **best subset** model are a subset of the $(k+1)$-variable **best subset** model | **False** |

**Reasoning.**

- **i (True).** Forward stepwise builds *up*: the $(k+1)$-variable model is formed by adding one predictor
  to the $k$-variable model and never removes anything. Nesting is guaranteed by the algorithm.
- **ii (True).** Backward stepwise builds *down*: the $k$-variable model is formed by deleting one
  predictor from the $(k+1)$-variable model. Nesting is again guaranteed, read in the reverse direction.
- **iii and iv (False).** Forward and backward are two independent greedy searches starting from opposite
  ends of the model space. Nothing links their paths, so no subset relationship is guaranteed.
- **v (False).** Best subset re-optimizes from scratch at every size. The optimal 2-variable model need not
  contain the single best predictor, because two moderately useful correlated predictors can jointly beat
  the strongest individual predictor. Nesting is therefore not guaranteed.

---
## Empirical verification

The claims above are structural, so a simulation can confirm them. We build a design in which two
moderately predictive but jointly complementary variables outperform the single strongest predictor,
which is exactly the situation that breaks the nesting property in statement (v).

In [1]:
import numpy as np
import pandas as pd
from itertools import combinations
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(8555)   # seed tied to the course number
n, p = 200, 6

# A design built to expose the nesting question in statement (v).
#   X2 and X3 jointly generate the response exactly.
#   X1 is a noisy proxy for their sum, so it is the strongest SINGLE predictor
#   but is redundant once X2 and X3 are both available.
X2 = rng.normal(size=n)
X3 = rng.normal(size=n)
X1 = (X2 + X3) / np.sqrt(2) + 0.30 * rng.normal(size=n)
y  = X2 + X3 + rng.normal(scale=0.05, size=n)
extra = [rng.normal(size=n) for _ in range(p - 3)]      # X4-X6 are pure noise

cols = [f"X{j+1}" for j in range(p)]
Xdf  = pd.DataFrame(np.column_stack([X1, X2, X3] + extra), columns=cols)

def rss(features):
    features = list(features)
    if not features:
        return float(((y - y.mean())**2).sum())
    m = LinearRegression().fit(Xdf[features], y)
    return float(((y - m.predict(Xdf[features]))**2).sum())

print(f"n = {n}, p = {p}")
print(f"corr(X1, y) = {np.corrcoef(X1, y)[0,1]:.3f}  <- strongest single predictor")
print(f"corr(X2, y) = {np.corrcoef(X2, y)[0,1]:.3f}")
print(f"corr(X3, y) = {np.corrcoef(X3, y)[0,1]:.3f}")

n = 200, p = 6
corr(X1, y) = 0.950  <- strongest single predictor
corr(X2, y) = 0.644
corr(X3, y) = 0.713


In [2]:
# ---- Best subset: exhaustive search at every size ----
best_subset = {}
for k in range(0, p+1):
    best = min(combinations(cols, k), key=rss) if k > 0 else ()
    best_subset[k] = (list(best), rss(best))

# ---- Forward stepwise ----
forward, cur = {}, []
forward[0] = ([], rss([]))
for k in range(1, p+1):
    cand = min((c for c in cols if c not in cur), key=lambda c: rss(cur+[c]))
    cur = cur + [cand]
    forward[k] = (cur.copy(), rss(cur))

# ---- Backward stepwise ----
backward, cur = {}, cols.copy()
backward[p] = (cur.copy(), rss(cur))
for k in range(p-1, -1, -1):
    drop = min(cur, key=lambda c: rss([x for x in cur if x != c]))
    cur = [x for x in cur if x != drop]
    backward[k] = (cur.copy(), rss(cur))

tbl = pd.DataFrame({
    "k": range(p+1),
    "BestSubset_RSS":  [round(best_subset[k][1], 2) for k in range(p+1)],
    "Forward_RSS":     [round(forward[k][1], 2)     for k in range(p+1)],
    "Backward_RSS":    [round(backward[k][1], 2)    for k in range(p+1)],
})
print(tbl.to_string(index=False))

 k  BestSubset_RSS  Forward_RSS  Backward_RSS
 0          345.86       345.86        345.86
 1           33.99        33.99        169.92
 2            0.49        30.53          0.49
 3            0.48         0.48          0.48
 4            0.48         0.48          0.48
 5            0.48         0.48          0.48
 6            0.48         0.48          0.48


### Verifying (a): best subset never has larger training RSS

In [3]:
ok_a   = all(best_subset[k][1] <= min(forward[k][1], backward[k][1]) + 1e-9 for k in range(p+1))
str_f  = any(best_subset[k][1] <  forward[k][1]  - 1e-9 for k in range(p+1))
str_b  = any(best_subset[k][1] <  backward[k][1] - 1e-9 for k in range(p+1))
print("(a) Best subset RSS <= both stepwise methods at every k :", ok_a)
print("    Strictly better than FORWARD  for some k            :", str_f)
print("    Strictly better than BACKWARD for some k            :", str_b)

(a) Best subset RSS <= both stepwise methods at every k : True
    Strictly better than FORWARD  for some k            : True
    Strictly better than BACKWARD for some k            : True


### Verifying (c): nesting properties

In [4]:
def nested(small, big):
    return set(small).issubset(set(big))

i_ok  = all(nested(forward[k][0],  forward[k+1][0])  for k in range(p))
ii_ok = all(nested(backward[k][0], backward[k+1][0]) for k in range(p))
iii   = all(nested(backward[k][0], forward[k+1][0])  for k in range(p))
iv    = all(nested(forward[k][0],  backward[k+1][0]) for k in range(p))
v     = all(nested(best_subset[k][0], best_subset[k+1][0]) for k in range(p))

print(f"i.   forward  k  subset of forward  k+1 : {i_ok}   (expected True)")
print(f"ii.  backward k  subset of backward k+1 : {ii_ok}  (expected True)")
print(f"iii. backward k  subset of forward  k+1 : {iii}  (expected False in general)")
print(f"iv.  forward  k  subset of backward k+1 : {iv}  (expected False in general)")
print(f"v.   best     k  subset of best     k+1 : {v}  (expected False in general)")

print("\nBest subset selections by size (note where nesting breaks):")
for k in range(0, 4):
    print(f"  k={k}: {best_subset[k][0]}")

i.   forward  k  subset of forward  k+1 : True   (expected True)
ii.  backward k  subset of backward k+1 : True  (expected True)
iii. backward k  subset of forward  k+1 : False  (expected False in general)
iv.  forward  k  subset of backward k+1 : False  (expected False in general)
v.   best     k  subset of best     k+1 : False  (expected False in general)

Best subset selections by size (note where nesting breaks):
  k=0: []
  k=1: ['X1']
  k=2: ['X2', 'X3']
  k=3: ['X2', 'X3', 'X4']


### Interpretation of the simulation

The RSS table confirms **(a)**. Best subset is never worse than either stepwise method, and here it is
*strictly* better than forward stepwise at $k=2$ and strictly better than backward stepwise at $k=1$.
Forward stepwise is trapped: having committed to the strong single proxy $X_1$ at step one, it cannot
reach the far superior pair $\{X_2, X_3\}$, which drives RSS essentially to zero.

The nesting checks confirm **(c)**. Statements i and ii return `True` because forward and backward
stepwise are nested by construction. Statement **v returns `False`**, exactly as the theory predicts: the
best one-variable model selects $X_1$, but the best two-variable model discards it in favour of
$\{X_2, X_3\}$, so the size-1 model is *not* contained in the size-2 model. Statements iii and iv also
return `False`, since forward and backward searches follow unrelated paths.

One caveat on method: a single simulation can only ever *falsify* a nesting claim, never prove one. A
design in which all three methods happen to agree would return `True` for every statement without making
iii, iv, or v generally true. The design above was therefore constructed so that redundancy among
correlated predictors makes the counterexample visible.

This also illustrates the trade-off emphasized in the module. Best subset is optimal on training RSS but
costs $2^p$ fits, whereas stepwise needs only $1 + p(p+1)/2$. Because none of the three guarantees the
lowest *test* error, final model choice should rest on cross-validation or a penalized criterion such as
$C_p$, AIC, or BIC.

---
## Reference

James, G., Witten, D., Hastie, T., Tibshirani, R., & Taylor, J. (2023). *An introduction to statistical
learning with applications in Python*. Springer. https://doi.org/10.1007/978-3-031-38747-0